# NAC coloring search

In this notebook we provide utilities to run benchmarks, analyze results and experiment with our code.

First we provide class for loading graph classes,
then a simple function for measuring performance of listing all NAC-colorings on a graph class.
Then we define how strategies are passed to our algorithm,
and after a framework for defining and running benchmarks.
Lastly, we provide tools for quick results analysis.

Many utility functions were moved from the notebook
into a separate file to improve clarity, see `benchmarks/notebook_utils.py`.

Make sure the `nac` directory is in your working directory, and that you installed `requierements.txt` into your virtual environment.
Also, make sure to uncompressed files in `benchmarks/precomputed` if you want to analyze our result yourself.

If you are using VScode, add this option to your `.vscode/settings.json` file.
```json
{
    "jupyter.notebookFileRoot": "${workspaceFolder}"
}
```
Otherwise, make sure your working directory is set correctly.

In [ ]:
from typing import *
from collections import defaultdict, deque
import random
import importlib

import numpy as np
import pandas as pd
import networkx as nx
import os
import time
import datetime
import itertools

from tqdm import tqdm

import nac as nac
from nac import NACValidClassType

import benchmarks
from benchmarks import datasets
from benchmarks import generators
import benchmarks.notebook_utils
from benchmarks.notebook_utils import *

seed=42
BENCHMARKS=False
ANALYTICS=True
SEARCH=False

In [ ]:
importlib.reload(nac)
importlib.reload(benchmarks)
importlib.reload(benchmarks.datasets)
importlib.reload(benchmarks.generators)
importlib.reload(benchmarks.notebook_utils)

_BENCH_FILE_PREFIX_V2 = "bench_res_v2"
_BENCH_FILE_PREFIX_V3 = "bench_res_v3"
_BENCH_FILE_PREFIX_V4 = "bench_res_v4"
_BENCH_FILE_PREFIX_V5 = "bench_res_v5"

OUTPUT_DIR = os.path.join("benchmarks", "runs_paper")
# OUTPUT_DIR = os.path.join("benchmarks", "precomputed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

benchmarks.notebook_utils.OUTPUT_DIR = OUTPUT_DIR
benchmarks.notebook_utils.OUTPUT_BENCH_FILE_START = _BENCH_FILE_PREFIX_V5
benchmarks.notebook_utils.OUTPUT_VERBOSE = False

In [ ]:
# Uncoment and restart notebook to output LaTeX plots.
# Then, comment and restart the notebook again.
# enable_latex_output()

# Show cumulative sums of graphs (as for larger amouns, the number of graphs is often smaller)
set_cum_sum(enabled=True)

# If there are two round, we may let (False) the run use combined limit of multiple runs. Less round then actually happen.
set_precise_rounds(enabled=False) # TODO revert

# Loading locally stored graphs

In [ ]:
class Graphs:
    """
    Randomly generated laman graphs of various sizes.
    """
    minimally_rigid_random = LazyList(lambda: datasets.load_minimally_rigid_random_graphs())
    """
    Graphs with no 3 nor 4 cyclesa.
    """
    no_3_nor_4_cycles = LazyList(lambda: datasets.load_no_3_nor_4_cycle_graphs())
    """
    Randomly generated globally rigid graphs using a threshold function.
    """
    globally_rigid_threshold = LazyList(lambda: datasets.load_globally_rigid_threshold_graphs())
    """
    Randomly generated globally rigid and NAC-critical graphs.
    These graphs should have just few or no NAC-colorings.
    """
    globally_rigid_nac_critical = LazyList(lambda: datasets.load_globally_rigid_nac_critical_graphs())
    """
    Graphs gathered from other cathegories that have no NAC-coloring and more than one △-connected component
    """
    no_NAC_coloring_gathered = LazyList(lambda: datasets.load_no_NAC_coloring_graphs_gathered())
    """
    Randomly generated NAC-critical graphs with at least n/4 △-connected components
    """
    nac_critical = LazyList(lambda: datasets.load_nac_critical_graphs())
    """
    Random (globally rigid) graphs that have no NAC-coloring and more than 2*sqrt(n) △-connected components
    """
    no_NAC_coloring_generated_40 = LazyList(lambda: datasets.load_no_NAC_coloring_graphs_generated(40))
    no_NAC_coloring_generated_50 = LazyList(lambda: datasets.load_no_NAC_coloring_graphs_generated(50))
    no_NAC_coloring_generated_60 = LazyList(lambda: datasets.load_no_NAC_coloring_graphs_generated(60))
    no_NAC_coloring_generated_70 = LazyList(lambda: datasets.load_no_NAC_coloring_graphs_generated(70))
    no_NAC_coloring_generated_80 = LazyList(lambda: datasets.load_no_NAC_coloring_graphs_generated(80))
    no_NAC_coloring_generated_90 = LazyList(lambda: datasets.load_no_NAC_coloring_graphs_generated(90))
    no_NAC_coloring_generated_100 = LazyList(lambda: datasets.load_no_NAC_coloring_graphs_generated(100))
    no_NAC_coloring_generated_110 = LazyList(lambda: datasets.load_no_NAC_coloring_graphs_generated(110))
    no_NAC_coloring_generated_120 = LazyList(lambda: datasets.load_no_NAC_coloring_graphs_generated(120))
    no_NAC_coloring_generated_130 = LazyList(lambda: datasets.load_no_NAC_coloring_graphs_generated(130))

    """
    Loads all the Laman graphs of the given size, pregenerated files allow the range of [5, 11]
    In case you want to use it in benchmarks, list all the graphs first.
    """
    def load_all_laman(vertex_num: int) -> Iterator[nx.Graph]:
        return datasets.load_minimally_rigid_all(vertices_num=vertex_num)

### File storage management

In [ ]:
def new_DataFrame(data: List[MeasurementResult] = []) -> pd.DataFrame:
    return pd.DataFrame(
        [x.to_list() for x in data],
        columns=COLUMNS,
    )

def update_stored_data(dfs: List[pd.DataFrame] = [], head_loaded: bool = True) -> pd.DataFrame:
    df = load_records()
    if head_loaded:
        display(df)
    if len(dfs) != 0:
        df = pd.concat((df, pd.concat(dfs)))
    df = df.drop_duplicates(subset=Columns.identifying, keep='last')
    store_results(df)
    return df

def migrate_v2_to_v3(dir: str = OUTPUT_DIR) -> pd.DataFrame:
    file_name_v2 = find_latest_record_file(_BENCH_FILE_PREFIX_V2, dir)
    path = os.path.join(dir, file_name_v2)
    df = pd.read_csv(path)
    df["use_smart_split"] = True
    df["used_monochromatic_classes"] = True
    df.loc[df["dataset"] == 'laman_random_no_smart_split', "use_smart_split"] = False
    df.loc[df["dataset"] == 'laman_random_no_smart_split', "dataset"] = 'laman_random'
    df = df[COLUMNS]
    store_results(df, None, dir)

def migrate_v3_to_v4(dir: str = OUTPUT_DIR) -> pd.DataFrame:
    file_name_v3 = find_latest_record_file(_BENCH_FILE_PREFIX_V3, dir)
    path = os.path.join(dir, file_name_v3)
    df = pd.read_csv(path)
    df["timestamp"] = datetime.datetime(1970, 1, 1)
    df["nac_first_merge"] = -1
    df["nac_first_merge_no_common_vertex"] = -1
    df["nac_all_merge"] = -1
    df["nac_all_merge_no_common_vertex"] = -1
    df = df[COLUMNS]
    store_results(df, None, dir)

def migrate_v4_to_v5(dir: str = OUTPUT_DIR) -> pd.DataFrame:
    file_name_v4 = find_latest_record_file(_BENCH_FILE_PREFIX_V4, dir)
    path = os.path.join(dir, file_name_v4)
    df = pd.read_csv(path)
    df = df.rename(columns = {
        "vertex_no": "vertex_num",
        "edge_no": "edge_num",
        "triangle_components_no": "triangle_components_num",
        "monochromatic_classes_no": "extended_classes_num",
        "nac_first_coloring_no": "nac_first_coloring_num",
        "nac_all_coloring_no": "nac_all_coloring_num",
        "used_monochromatic_classes": "used_extended_classes",
    })
    store_results(df, None, dir)

## Utils for benchmarks

See `../NAC_presentation.ipynb` for further description.

In [ ]:
class Promising:
    RELABELING = [
        "none",
        # "random",
    ]
    SPLITTING = [
        "none",
        # "cycles_match_chunks",
        "neighbors",
        "neighbors_degree",
    ]
    MERGE = [
        "linear",
        "shared_vertices",
        # "log",
        # "min_max",
        # "score",
        # "sorted_size",
    ]
    SIZES = [5]

    strategies = list(itertools.product(
        RELABELING, SPLITTING, MERGE, SIZES,
    ))
print(f"Strategies:  {len(Promising.strategies)}")

In [ ]:
def construct_subgraph_algo_name(param: Tuple[str, str, str, int], use_smart_split: bool) -> str:
    relabel, split, merge, subgraph = param
    algo_name = "subgraphs-{}-{}-{}{}".format(
        merge, split, subgraph, "-smart" if use_smart_split else ""
    )
    return algo_name

In [ ]:
def measure_for_graph_class(
    dataset_name: str,
    graphs: Iterable[nx.Graph],
    graph_timeout: int,
    all_max_classes_num: int = 28,
    rounds: int = 2,
    allow_naive: bool = True,
    use_smart_split: bool = False,
    use_triangle_extended_classes: bool = True,
    df_seen: pd.DataFrame | None | Callable[[], pd.DataFrame] = load_records,
    save_every: int | None = 15*60,
    verbose: bool = False,
) -> pd.DataFrame:
    """
    Runs benchmarks for the given graph class.

    Parameters:
        dataset_name: Name of the dataset stored in the output csv
        graphs: Iterable of graphs to benchmark
        graph_timeout: Timeout for each graph in seconds
        all_max_classes_num: Maximum number of NAC-valid classes (based on use_triangle_extended_classes) to search for all NAC-colorings
        rounds: Number of rounds to run for each graph
        allow_naive: Whether to run the naive algorithm
        use_smart_split: Whether to use smart split
        use_triangle_extended_classes: Whether to use monochromatic classes or tiriangle connected components
        df_seen: Dataframe with already measured data, so already tried graphs and strategies can be skipped
        save_every: save progress every number of seconds
    """
    if callable(df_seen):
        df_seen = df_seen()

    dataset_name = dataset_name.replace(" ", "_").lower()
    if df_seen is None:
        df_seen = new_DataFrame()
    df_seen = df_seen.query(f"{Columns.DATASET} == '{dataset_name}'")
    df_seen.set_index("graph", inplace=True)

    results: List[MeasurementResult] = []
    all_results: List[MeasurementResult] = []

    last_save = time.time()

    for graph in tqdm(graphs):
        # this would be a functin if python would not have broken scoping
        if save_every is not None and len(results) > 0:
            now = time.time()
            if now - last_save > save_every:
                all_results.extend(results)
                df = new_DataFrame(results)
                update_stored_data([df], head_loaded=False)
                results = []
                last_save = now


        triangle_component_num = len(nac.find_nac_mono_classes(graph=graph, class_type=NACValidClassType.TRIANGLES)[1])
        extended_classes_num = len(nac.find_nac_mono_classes(graph=graph, class_type=NACValidClassType.EXTENDED)[1])
        if use_triangle_extended_classes:
            first_only = all_max_classes_num < extended_classes_num
        else:
            first_only = all_max_classes_num < triangle_component_num

        strategies = Promising.strategies
        if allow_naive:
            strategies = itertools.chain(strategies, (None,))

        graph_id = graph_to_id(graph)
        if graph_id in df_seen.index:
            df_graph = df_seen.loc[graph_id]
        else:
            df_graph = df_seen.iloc[:0]

        for strategy in strategies:
            # skip test that already run
            if strategy is not None:
                relabel = strategy[0]
                algorithm = construct_subgraph_algo_name(strategy, use_smart_split=use_smart_split)
                prev_record = df_graph.query(
                    f"{Columns.RELABEL} == '{strategy[0]}'"
                    + f" and {Columns.SPLIT} == '{strategy[1]}'"
                    + f" and {Columns.MERGING} == '{strategy[2]}'"
                    + f" and {Columns.SUBGRAPH_SIZE} == {strategy[3]}"
                    + f" and {Columns.USE_SMART_SPLIT} == {use_smart_split}"
                    + f" and {Columns.USED_EXTENDED_CLASSES} == {use_triangle_extended_classes}"
                )
            else:
                relabel = "none"
                algorithm="cycles"
                prev_record = df_graph.query(
                    f"{Columns.RELABEL} == 'none'"
                    + f" and {Columns.SPLIT} == 'naive-cycles'"
                    + f" and {Columns.MERGING} == 'naive-cycles'"
                    + f" and {Columns.SUBGRAPH_SIZE} == 0"
                    + f" and {Columns.USE_SMART_SPLIT} == {use_smart_split}"
                    + f" and {Columns.USED_EXTENDED_CLASSES} == {use_triangle_extended_classes}"
                )
            if len(prev_record) > 0:
                # ensureds graphs are recomputed if all_max_vertex_num is increased
                if first_only or list(prev_record[Columns.ALL_MEAN_TIME])[-1] > 0:
                    continue

            try:
                search_res = nac_benchmark_core(
                    graph,
                    rounds=rounds,
                    first_only=first_only,
                    algorithm=algorithm,
                    relabel_strategy=relabel,
                    use_triangle_extended_classes=use_triangle_extended_classes,
                    time_limit=graph_timeout,
                )

                relabel, split, merge, subgraph_size = strategy if strategy else ("none", "naive-cycles", "naive-cycles", 0)
                res = create_measurement_result(
                    graph=graph,
                    dataset_name=dataset_name,
                    triangle_components=triangle_component_num,
                    extended_classes=extended_classes_num,
                    nac_first=search_res.first,
                    nac_all=search_res.all,
                    relabel_strategy=relabel,
                    split_strategy=split,
                    merge_strategy=merge,
                    subgraph_size=subgraph_size,
                    use_smart_split=use_smart_split,
                    used_triangle_extended_classes=use_triangle_extended_classes,
                    timeout_milliseconds=graph_timeout * 1000,
                )
                results.append(res)
                if (verbose):
                    print(f"Strategy {strategy} took {res.nac_first_mean_time} ms")
            except Exception as e:
                print(f"Exception for strategy {strategy}: {e}")
                # raise e

    all_results.extend(results)
    if len(all_results) == 0:
        print("All runs skipped")

    if len(results) > 0:
        df = new_DataFrame(results)
        update_stored_data([df], head_loaded=False)

    df = new_DataFrame(all_results)
    df = df.sort_values(by=[Columns.ALL_MEAN_TIME, Columns.FIRST_MEAN_TIME])
    return df

# Running benchmarks

### Minimally Rigid - Random

In [ ]:
if BENCHMARKS:
    df_laman_random = measure_for_graph_class(
        "Minimally rigid random",
        Graphs.minimally_rigid_random,
        graph_timeout=3,
    )

### No 3 nor 4 cycles

In [ ]:
if BENCHMARKS:
    display(pd.Series([g.number_of_nodes() for g in Graphs.no_3_nor_4_cycles]).value_counts())
    df_no_3_nor_4_cycles = measure_for_graph_class(
        "No 3 nor 4 cycles",
        Graphs.no_3_nor_4_cycles,
        all_max_classes_num=20,
        graph_timeout=3,
    )
display(max(Graphs.no_3_nor_4_cycles, key=lambda g: g.number_of_nodes()).number_of_nodes())

### Globally rigid threshold

In [ ]:
if BENCHMARKS:
    measure_for_graph_class(
        "Globally rigid threshold",
        Graphs.globally_rigid_threshold,
        graph_timeout=5,
    )

### NAC critical

In [ ]:
if BENCHMARKS:
    measure_for_graph_class(
        "NAC critical",
        Graphs.nac_critical, # 6,5k/16k graphs
        graph_timeout=5,
        all_max_classes_num=0,
        use_triangle_extended_classes=False,
        allow_naive=False,
    )

### No NAC coloring

In [ ]:
if BENCHMARKS:
    for name, df in {
        "no_NAC_coloring_generated_40": Graphs.no_NAC_coloring_generated_40,
        "no_NAC_coloring_generated_50": Graphs.no_NAC_coloring_generated_50,
        "no_NAC_coloring_generated_60": Graphs.no_NAC_coloring_generated_60,
        "no_NAC_coloring_generated_70": Graphs.no_NAC_coloring_generated_70,
        "no_NAC_coloring_generated_80": Graphs.no_NAC_coloring_generated_80,
        "no_NAC_coloring_generated_90": Graphs.no_NAC_coloring_generated_90,
        "no_NAC_coloring_generated_100": Graphs.no_NAC_coloring_generated_100,
        "no_NAC_coloring_generated_110": Graphs.no_NAC_coloring_generated_110,
        "no_NAC_coloring_generated_120": Graphs.no_NAC_coloring_generated_120,
        "no_NAC_coloring_generated_130": Graphs.no_NAC_coloring_generated_130,
    }.items():
        for smart_split in [False]:
        # for smart_split in [False, True]:
            print(name, smart_split)
            measure_for_graph_class(
                name,
                df[:500],
                graph_timeout=15,
                all_max_classes_num=0,
                allow_naive=False,
                use_triangle_extended_classes=False,
                use_smart_split=smart_split,
            )

# Analytics

See `../NAC_presentation.ipynb` for further details.

In [ ]:
df_analytics_loaded = load_records()
df_analytics_loaded.set_index(Columns.GRAPH, inplace=True)
df_analytics_loaded = df_analytics_loaded.query(f"{Columns.DATASET} != 'test'")
display(df_analytics_loaded.columns)
display(list(df_analytics_loaded[Columns.DATASET].unique()))
display(list(df_analytics_loaded[Columns.RELABEL].unique()))
display(list(df_analytics_loaded[Columns.SPLIT].unique()))
display(list(df_analytics_loaded[Columns.MERGING].unique()))

# Transform
df_analytics_loaded = df_analytics_loaded.assign(split_merging=lambda x: (x[Columns.SPLIT] + " & " + x[Columns.MERGING]).str.replace("naive-cycles & naive-cycles", "naive cycles").str.replace("&", r"\&").str.replace("_", r"\_"))
df_analytics_loaded = df_analytics_loaded.assign(split_merging_smart=lambda x: x["split_merging"] + " & " + x[Columns.USE_SMART_SPLIT].astype(str))
df_analytics_loaded.sort_values(by=Columns.SPLIT, inplace=True, kind="stable") # to make graph colors more consistent
df_analytics_loaded.sort_values(by=Columns.MERGING, inplace=True, kind="stable")

In [ ]:
# Preserver the original data
df_analytics = df_analytics_loaded

# Drop runs where only a single run passed
df_analytics = df_analytics.query(f"{Columns.FIRST_ROUNDS} == 2 and ({Columns.ALL_ROUNDS} == 2 or {Columns.ALL_MEAN_TIME} == 0)")

# Filter out trivial graphs
before_trivial_drop = len(df_analytics)
df_analytics = df_analytics.query(f"({Columns.EXTENDED_CLASSES_NUM} > 1 and {Columns.USED_EXTENDED_CLASSES} == True) or ({Columns.TRIANGLE_COMPONENTS_NUM} > 1 and {Columns.USED_EXTENDED_CLASSES} == False)")
print(f"Dropped {before_trivial_drop - len(df_analytics)} trivial (single class) graphs.")


# Filter bad strategies
df_with_failing = df_analytics
df_analytics = df_analytics.query(f"{Columns.SPLIT} == 'naive-cycles' or {Columns.SPLIT} == 'none' or {Columns.SPLIT} == 'neighbors' or {Columns.SPLIT} == 'neighbors_degree'") # or {Columns.SPLIT} == 'cycles_match_chunks'
df_analytics = df_analytics.query(f"{Columns.MERGING} == 'naive-cycles' or {Columns.MERGING} == 'linear' or {Columns.MERGING} == 'shared_vertices'") #  or {Columns.MERGING} == 'score'

# Graphs with no NAC coloring and more triangle connected components
df_analytics_no_nac = df_analytics.query(f"{Columns.FIRST_COLORING_NUM} == 0 and {Columns.TRIANGLE_COMPONENTS_NUM} > 1 and {Columns.USED_EXTENDED_CLASSES} == False")

# Statistics
def analyze_general(df: pd.DataFrame) -> None:
    print(f"Total runs: {len(df)}", )
    print(f"Total graphs: {len(df.index.unique())}")
    df_finished = df.query(f"{Columns.ANY_FINISHED} == True")
    df_failed = df.query(f"{Columns.ANY_FINISHED} == False")
    print(f"Runs that did/not/finish: {len(df_finished)}/{len(df_failed)}/{len(df)} ({np.round(len(df_failed)/len(df)*100, 1)}% did not finish)")
    print(f"Graphs where some runs did/not/finish: {df_finished.index.nunique()}/{df_failed.index.nunique()}/{df.index.nunique()}")

def analyze_colorings(df: pd.DataFrame) -> None:
    df_finished = df.query(f"{Columns.ANY_FINISHED} == True")
    print(f"Graphs with  a NAC-coloring:", df_finished.query(f"{Columns.FIRST_COLORING_NUM}  > 0").index.nunique())
    print(f"Graphs with no NAC-coloring:", df_finished.query(f"{Columns.FIRST_COLORING_NUM} == 0").index.nunique())
    print(f"Graphs with no NAC-coloring and more monochromatic classes:", df_finished.query(f"{Columns.FIRST_COLORING_NUM} == 0 and {Columns.EXTENDED_CLASSES_NUM} > 1").index.nunique())

def analyze_finished(df: pd.DataFrame) -> None:
    graphs = df.index.unique()
    graphs_all_finished = filter_graphs_that_finished_for_all_strategies(df)
    graphs_nonnaive_finished = filter_graphs_that_finished_for_all_strategies(df.query(f"{Columns.SPLIT} != 'naive-cycles'"))
    print(f"{len(graphs_all_finished)}/{len(graphs)} graphs finished on all tested strategies.")
    print(f"{len(graphs_nonnaive_finished)}/{len(graphs)} graphs finished on all tested strategies excluding naive cycles.")

    df_all_finished = df.loc[graphs_all_finished]
    df_nonnaive_finished = df.loc[graphs_nonnaive_finished]
    print(f"Records corresponding to graph, that finished on all tested strategies: {len(df_all_finished)}")
    print(f"Records corresponding to graph, that finished on all tested strategies excluding naive cycles: {len(df_nonnaive_finished)}")


print("All together:")
analyze_general(df_with_failing)
print()

print("Just preffered strategies:")
analyze_general(df_analytics)
analyze_colorings(df_analytics)
print()

analyze_finished(df_analytics)

df_analytics = replace_failed_results(df_analytics)

## Graph classes

In [ ]:
for dataset in ["nac_critical", "globally_rigid_threshold", "minimally_rigid_random"]:
    fig = plot_extended_vs_original_triangle_components(df_analytics, dataset=dataset)
    display(fig)
    export_extended_classes_vs_triangle_components(fig, dataset)

### Minimally rigid - Random

In [ ]:
if ANALYTICS:
    title = 'Minimally rigid'
    dataset_name = 'minimally_rigid_random'

    dataset = finished_graphs(df_analytics.query(f"{Columns.DATASET} == '{dataset_name}'"))
    figs = [fig for fig in plot_frame(title, dataset, ops_value_columns_sets=Columns.first)]
    [display(fig) for fig in figs]
    export_standard_figure_list(dataset_name, figs)

    dataset = df_analytics.query(f"{Columns.DATASET} == '{dataset_name}' and {Columns.EXTENDED_CLASSES_NUM} <= 28")
    figs = [fig for fig in plot_frame(title, dataset, ops_value_columns_sets=Columns.all)]
    [display(fig) for fig in figs]
    export_standard_figure_list(dataset_name, figs)

### No 3 nor 4 cycles

In [ ]:
if ANALYTICS:
    title = 'No 3 nor 4 cycles'
    dataset_name = 'no_3_nor_4_cycles'

    dataset = finished_graphs(df_analytics.query(f"{Columns.DATASET} == '{dataset_name}'"))
    figs = [fig for fig in plot_frame(title, dataset, ops_value_columns_sets=Columns.first)]
    [display(fig) for fig in figs]
    export_standard_figure_list(dataset_name, figs)

    dataset = finished_graphs_no_naive(df_analytics.query(f"{Columns.DATASET} == '{dataset_name}'"))
    figs = [fig for fig in plot_frame(title, dataset, ops_value_columns_sets=Columns.all)]
    [display(fig) for fig in figs]
    export_standard_figure_list(dataset_name, figs)

### Globally Rigid (threshold)

In [ ]:
if ANALYTICS:
    title = 'Globally rigid (threshold)'
    dataset_name = 'globally_rigid_threshold'

    dataset = finished_graphs(df_analytics.query(
        f"{Columns.DATASET} == '{dataset_name}' and {Columns.EXTENDED_CLASSES_NUM} <= 28")
    )
    figs = [fig for fig in plot_frame(title, dataset,
        ops_value_columns_sets=Columns.first,
        ops_x_column=[Columns.EXTENDED_CLASSES_NUM],
    )]
    [display(fig) for fig in figs]
    export_standard_figure_list(dataset_name, figs)

    dataset = finished_graphs_no_naive(df_analytics.query(f"{Columns.DATASET} == '{dataset_name}'"))
    figs = [fig for fig in plot_frame(title, dataset,
        ops_value_columns_sets=Columns.all,
        ops_x_column=[Columns.EXTENDED_CLASSES_NUM],
    )]
    [display(fig) for fig in figs]
    export_standard_figure_list(dataset_name, figs)

### Globally Rigid (NAC critical)

In [ ]:
if ANALYTICS:
    title = 'Globally rigid (NAC critical)'
    dataset_name = 'globally_rigid_nac_critical'

    dataset = finished_graphs(df_analytics.query(f"{Columns.DATASET} == '{dataset_name}' and {Columns.EXTENDED_CLASSES_NUM} <= 25"))
    figs = [fig for fig in plot_frame(title, dataset,
        ops_value_columns_sets=Columns.first,
        ops_x_column=[Columns.EXTENDED_CLASSES_NUM],
    )]
    [display(fig) for fig in figs]
    export_standard_figure_list(dataset_name, figs)

    dataset = finished_graphs_no_naive(df_analytics.query(f"{Columns.DATASET} == '{dataset_name}'"))
    figs = [fig for fig in plot_frame(title, dataset,
        ops_value_columns_sets=Columns.all,
        ops_x_column=[Columns.EXTENDED_CLASSES_NUM],
    )]
    [display(fig) for fig in figs]
    export_standard_figure_list(dataset_name, figs)

In [ ]:
if ANALYTICS:
    title = 'Globally rigid (NAC critical)'
    dataset_name = 'globally_rigid_nac_critical_triangle_components'

    dataset = finished_graphs_no_naive(df_analytics.query(f"{Columns.DATASET} == '{dataset_name}'"))
    figs = [fig for fig in plot_frame(title, dataset,
        ops_value_columns_sets=Columns.first,
        ops_x_column=[Columns.EXTENDED_CLASSES_NUM],
    )]
    [display(fig) for fig in figs]
    export_standard_figure_list(dataset_name, figs)

In [ ]:
if ANALYTICS:
    title = 'Globally rigid (NAC critical + threshold)'
    dataset_name = 'globally_rigid_nac_critical_triangle_components'

    dataset = finished_graphs_no_naive(df_analytics.query(f"{Columns.DATASET} == '{dataset_name}' or {Columns.DATASET} == 'globally_rigid_threshold_triangle_components'"))
    figs = [fig for fig in plot_frame(title, dataset,
        ops_value_columns_sets=Columns.first,
        ops_x_column=[Columns.EXTENDED_CLASSES_NUM],
    )]
    [display(fig) for fig in figs]
    export_standard_figure_list(dataset_name, figs)

### NAC-critical

In [ ]:
if ANALYTICS:
    dataset_name = 'nac_critical'

    title = 'NAC-critical - no NAC coloring'
    dataset = finished_graphs(df_analytics.query(f"{Columns.DATASET} == '{dataset_name}' and {Columns.FIRST_COLORING_NUM} == 0"))
    figs = [fig for fig in plot_frame(title, dataset,
        ops_value_columns_sets=Columns.first,
        ops_x_column=[Columns.TRIANGLE_COMPONENTS_NUM],
    )]
    [display(fig) for fig in figs]
    export_standard_figure_list(dataset_name + "_none", figs)

    title = 'NAC-critical - some NAC coloring'
    dataset = finished_graphs(df_analytics.query(f"{Columns.DATASET} == '{dataset_name}' and {Columns.FIRST_COLORING_NUM} > 0 and {Columns.TRIANGLE_COMPONENTS_NUM} <= 150"))
    figs = [fig for fig in plot_frame(title, dataset,
        ops_value_columns_sets=Columns.first,
        ops_x_column=[Columns.TRIANGLE_COMPONENTS_NUM],
    )]
    [display(fig) for fig in figs]
    export_standard_figure_list(dataset_name + "_some", figs)

### No NAC-coloring - generated

In [ ]:
def query_no_nac_coloring_generated(base: pd.DataFrame) -> pd.DataFrame:
    base = base.reset_index(inplace=False)
    base = base.query(f"{Columns.SPLIT} != 'naive-cycles'")
    base_40 = base.query(f"{Columns.DATASET} == 'no_nac_coloring_generated_40'")
    base_50 = base.query(f"{Columns.DATASET} == 'no_nac_coloring_generated_50'")
    base_60 = base.query(f"{Columns.DATASET} == 'no_nac_coloring_generated_60'")
    base_70 = base.query(f"{Columns.DATASET} == 'no_nac_coloring_generated_70'")
    base_80 = base.query(f"{Columns.DATASET} == 'no_nac_coloring_generated_80'")
    base_90 = base.query(f"{Columns.DATASET} == 'no_nac_coloring_generated_90'")
    base_100 = base.query(f"{Columns.DATASET} == 'no_nac_coloring_generated_100'")
    base_110 = base.query(f"{Columns.DATASET} == 'no_nac_coloring_generated_110'")
    base_120 = base.query(f"{Columns.DATASET} == 'no_nac_coloring_generated_120'")
    base_130 = base.query(f"{Columns.DATASET} == 'no_nac_coloring_generated_130'")
    # df = base_40
    df = pd.concat([base_40, base_50, base_60, base_70, base_80, base_90, base_100, base_110, base_120, base_130], ignore_index=True)
    df.set_index(Columns.GRAPH, inplace=True)
    return df

In [ ]:
if ANALYTICS:
    title = 'No NAC-coloring, ▵-connected components'
    dataset_name = 'no_nac_coloring_generated'

    base = df_analytics_no_nac
    df = query_no_nac_coloring_generated(base)
    df = finished_graphs_no_naive(df)
    df = df.query(f"{Columns.TRIANGLE_COMPONENTS_NUM} >= 40 and {Columns.TRIANGLE_COMPONENTS_NUM} <= 250")

    figs = [fig for fig in plot_frame(
        title,
        df.query(f"({Columns.SUBGRAPH_SIZE}==4 or {Columns.SUBGRAPH_SIZE}==5)"),
        ops_x_column=[Columns.TRIANGLE_COMPONENTS_NUM],
        ops_based_on=["split_merging"],
        filter_out_exhaustive_mergin_strategies_for_first=False,
    )]
    [display(fig) for fig in figs]
    export_standard_figure_list(dataset_name, figs)

    figs = [fig for fig in plot_frame(
        title,
        df,
        ops_x_column=[Columns.TRIANGLE_COMPONENTS_NUM],
        ops_based_on=[Columns.SUBGRAPH_SIZE],
        filter_out_exhaustive_mergin_strategies_for_first=False,
    )]
    [display(fig) for fig in figs]
    export_standard_figure_list(dataset_name, figs)

## Other strategies

### Other - minimally rigid

In [ ]:
if ANALYTICS:
    title = 'Minimally rigid'
    dataset_name = 'minimally_rigid_random'

    dataset = drop_outliers(finished_graphs_no_naive(df_with_failing.query(f"{Columns.DATASET} == '{dataset_name}' and {Columns.SPLIT} == 'neighbors' and {Columns.MERGING} != 'score'")))
    figs = [fig for fig in plot_frame( title, dataset)]
    [display(fig) for fig in figs]
    export_standard_figure_list(dataset_name + "_failing_merging", figs)

    dataset = drop_outliers(finished_graphs_no_naive(df_with_failing.query(f"{Columns.DATASET} == '{dataset_name}' and {Columns.MERGING} == 'linear'")))
    figs = [fig for fig in plot_frame( title,dataset)]
    [display(fig) for fig in figs]
    export_standard_figure_list(dataset_name + "_failing_split", figs)

### Other - No 3 nor 4 cycles

In [ ]:
if ANALYTICS:
    title = 'No 3 nor 4 cycles'
    dataset_name = 'no_3_nor_4_cycles_new'
    dataset_export_name = 'no_3_nor_4_cycles'

    dataset = drop_outliers(finished_graphs_no_naive(df_with_failing.query(f"{Columns.DATASET} == '{dataset_name}' and {Columns.SPLIT} == 'neighbors' and {Columns.MERGING} != 'score'")))
    figs = [fig for fig in plot_frame( title, dataset)]
    [display(fig) for fig in figs]
    export_standard_figure_list(dataset_export_name + "_failing_merging", figs)

    dataset = drop_outliers(finished_graphs_no_naive(df_with_failing.query(f"{Columns.DATASET} == '{dataset_name}' and {Columns.MERGING} == 'linear'")))
    figs = [fig for fig in plot_frame( title,dataset)]
    [display(fig) for fig in figs]
    export_standard_figure_list(dataset_export_name + "_failing_split", figs)

### Other - globally rigid

In [ ]:
if ANALYTICS and False:
    title = 'Globally rigid'
    dataset_name = 'globally_rigid'

    dataset = drop_outliers(finished_graphs_no_naive(df_with_failing.query(f"{Columns.DATASET} == '{dataset_name}' and {Columns.SPLIT} == 'neighbors' and {Columns.MERGING} != 'score'")))
    figs = [fig for fig in plot_frame( title, dataset)]
    [display(fig) for fig in figs]
    export_standard_figure_list(dataset_name + "_failing_merging", figs)

    dataset = drop_outliers(finished_graphs_no_naive(df_with_failing.query(f"{Columns.DATASET} == '{dataset_name}' and {Columns.MERGING} == 'linear'")))
    figs = [fig for fig in plot_frame( title,dataset)]
    [display(fig) for fig in figs]
    export_standard_figure_list(dataset_name + "_failing_split", figs)

### Other - No NAC-coloring

In [ ]:
if ANALYTICS:
    base = df_with_failing.query(f"{Columns.SUBGRAPH_SIZE}==4 or {Columns.SUBGRAPH_SIZE}==5")
    df = query_no_nac_coloring_generated(base)
    title = 'No NAC-coloring'
    cut_at = 70 # maximun number of vertices where enough data is available

    dataset = drop_outliers(finished_graphs_no_naive(df.query(f"{Columns.SPLIT} == 'neighbors' and {Columns.VERTEX_NUM} < {cut_at}")))
    figs = [fig for fig in plot_frame(
        title, dataset,
        ops_x_column=[Columns.TRIANGLE_COMPONENTS_NUM],
        filter_out_exhaustive_mergin_strategies_for_first=False,
    )]
    [display(fig) for fig in figs]
    export_standard_figure_list("no_nac_coloring_generated_failing_merging", figs)

    dataset = drop_outliers(finished_graphs_no_naive(df.query(f"{Columns.MERGING} == 'linear' and {Columns.VERTEX_NUM} < {cut_at}")))
    figs = [fig for fig in plot_frame(
        title, dataset,
        ops_x_column=[Columns.TRIANGLE_COMPONENTS_NUM],
        filter_out_exhaustive_mergin_strategies_for_first=False,
    )]
    [display(fig) for fig in figs]
    export_standard_figure_list("no_nac_coloring_generated_failing_split", figs)

## Relative number of checks performed

In [ ]:
if ANALYTICS:
    data = df_analytics.query(
        f"({Columns.SPLIT} != 'naive-cycles') and ({Columns.USED_EXTENDED_CLASSES}==True) and {Columns.EXTENDED_CLASSES_NUM} <= 28"
    )
    data = data.query(f"{Columns.DATASET} == 'minimally_rigid_random'")
    figs = [fig for fig in plot_is_NAC_coloring_calls(data)]
    title = 'All datasets'
    dataset_name = 'check-comparision'
    [display(fig) for fig in figs]
    export_standard_figure_list(dataset_name, figs)

# Comparison with naive approach

In this section of benchmarks we run our algorithm on all the minimally 2-rigid graphs of specified size.
The goal is to show the performance improvement over the previous SOTA - naive approach.
For clarity and simplicity we use only a single strategy - `neighbors_degree` with `linear` merging.

In [ ]:
def chunked_iterable(iterable, size) -> Iterable[tuple[nx.Graph, ...]]:
    it = iter(iterable)
    while True:
        chunk = tuple(itertools.islice(it, size))
        if not chunk:
            break
        yield chunk

def find_colorings_on_all_graphs(
    dataset_name: str,
    graphs: Callable[[], Iterable[nx.Graph]],
    vertex_num: int,
    algorithm: str = "subgraphs-linear-neighbors_degree-4-smart",
    chunk_size: int = 10_000,
    DIR: str = os.path.join("benchmarks", "results", "iteration"),
    time_log: str | None =  None
):
    os.makedirs(DIR, exist_ok=True)
    time_log = time_log or os.path.join(DIR, "log.csv")

    stats = defaultdict(int)
    rand = random.Random(42)

    # print(f"Running {algorithm} on {len(graphs)} graphs with {vertex_no} vertices")
    print(f"Running {algorithm} on graphs with {vertex_num} vertices")
    graphs = graphs()

    total_time = 0.0
    progress_bar = tqdm(f"{algorithm} on {vertex_num} vertices")
    for graphs_chunk in chunked_iterable(graphs, chunk_size):
        start = time.time()
        for graph in graphs_chunk:
            iterable = nac.NAC_colorings(
                graph,
                algorithm=algorithm,
                relabel_strategy="none",
                use_decompositions=False,
                use_has_coloring_check=False,
                seed=rand.randint(0, 2**32 - 1),
            )
            counter = itertools.count()
            deque(zip(iterable, counter), maxlen=0)
            coloring_no = next(counter) // 2
            stats[coloring_no] += 1
            progress_bar.update(1)
        end = time.time()
        total_time += end - start
    progress_bar.close()

    print(f"The operation took {int(total_time)} s")
    with open(time_log, "a") as f:
        print(f"{vertex_num},{algorithm},{int(1000*total_time)}", file=f, flush=True)

    data = np.array(list(stats.items()))
    df = pd.DataFrame(data, columns=["coloring_cnt", "graph_cnt"])
    df.sort_values(by="coloring_cnt", inplace=True)
    df["graph_cnt"]
    print(
        f"Most colorings: {tuple(df.iloc[-1])}, Most common: {tuple(df.loc[df["graph_cnt"].idxmax()])} (coloring_cnt, graph_cnt)"
    )
    # print(df.tail(n=50))

    df.to_csv(os.path.join(DIR, f"{dataset_name}_{vertex_num}_{algorithm}.csv"))

In [ ]:
if False:
    for n in list(range(7, 12 + 1)):
        graphs = lambda: datasets.load_minimally_rigid_all(n)

        find_colorings_on_all_graphs("minimally_rigid", graphs, n,)
        find_colorings_on_all_graphs("minimally_rigid", graphs, n, "naive") if n <= 10 else None